In [1]:
import torch
import copy

from huggingface_hub import login
from datasets import load_dataset
from torch.nn import CrossEntropyLoss
from transformers import AutoModelForCausalLM, AutoTokenizer, DataCollatorWithPadding
from peft import get_peft_model, LoraConfig, get_peft_model_state_dict, set_peft_model_state_dict

from harness.datasets import Dreaddit, IRFDataset
from harness.dataclasses import FLClient, FLMessage
from harness.helpers import explode_conversations

/Users/hutchii/PycharmProjects/MFMoE/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [45]:
login()

In [139]:
smol_model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM3-3B")
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM3-3B")
collate_fn = DataCollatorWithPadding(tokenizer=tokenizer)

## TODO: Modify this for the ROBERTA model
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

Loading weights: 100%|██████████| 326/326 [00:00<00:00, 4053.98it/s]


In [140]:
# Init server model
server_model = copy.copy(smol_model)
base_peft_model = get_peft_model(server_model, peft_config=lora_config)

# This needs to be sent to each client to start federation.
global_adapters = get_peft_model_state_dict(base_peft_model)

In [147]:
# Dreaddit
STRESS_SUPPORT_STATEMENT = "It sounds like you are carrying a lot of stress and feeling completely exhausted. Please remember to be gentle with yourself today, and consider stepping away for a short break to rest."

GENERAL_SUPPORT_STATEMENT = "Thank you for sharing your thoughts today. I hope things continue to go smoothly for you and that you have a wonderful, peaceful day ahead!"

BASE_INSTRUCTION = "Analyze this Reddit post for signs of psychological stress. Provide a brief, supportive response in 1–2 sentences. If stress is present, acknowledge feelings and suggest one helpful action. If no stress, provide encouragement. Be empathetic but avoid medical advice. Post: "

ds = load_dataset("hutchii/dreaddit")

train = ds["train"]
prompts = train.map(build_dreaddit_features).select_columns(column_names=["prompt", "response"])
prompt_data = prompts.data.to_pylist()

dread_dataset = Dreaddit(texts=prompt_data, text_tokenizer=tokenizer)
dread_dl = DataLoader(dataset=dread_dataset, batch_size=32, collate_fn=custom_collate_fn)

dread_client: FLClient = FLClient(client_name="dread", data_loader=dread_dl)

In [148]:
#IRF
irf_dataset = load_dataset("hutchii/InterpersonalRiskFactors")
# belong:0 burden:0 - 0
# belong:1 burden:0 - 1
# belong:0 burden:1 - 2
# belong:1 burden:1 - 3

irf_tuning_statements = {
    "0":"Thank you for opening up and sharing your thoughts today. I hope things continue to go smoothly for you and that you have a peaceful, restful week ahead!",
    "1":"I hear how painful and exhausting it feels to be disconnected right now, but please know that your feelings are valid and you do not have to face this isolation alone. Consider reaching out to a trusted friend or community space when you feel ready.",
    "2":"It sounds like you are carrying a heavy emotional weight and feeling like a hardship to those around you, but your presence has genuine value. Please be gentle with yourself today and remember that needing support does not make you a burden.",
    "3":"I hear how completely overwhelming things feel right now, especially when experiencing both deep isolation and the feeling of being a burden to others. Please know that you matter and you do not have to carry this alone—consider connecting with a supportive person or crisis resource today."
}

irf_instruction = "Analyze this post for signs of interpersonal risk factors, specifically Thwarted Belongingness (feeling isolated, lonely, or disconnected) and Perceived Burdensomeness (feeling like a drain or weight on others). Provide a brief, supportive response in 1–2 sentences addressing these feelings. Be empathetic and encouraging while avoiding medical advice. Post: "

iff_train = irf_dataset["train"]
train_data = iff_train.map(build_irf_features).select_columns(column_names=["prompt", "response"]).data.to_pylist()

irf_dataset = IRFDataset(texts=train_data, tokenizer=tokenizer)
irf_dl = DataLoader(dataset=irf_dataset, batch_size=32, collate_fn=collate_fn)

irf_client:FLClient = FLClient(client_name="irf", data_loader=irf_dl)

In [149]:
clients: list[FLClient] = [dread_client, irf_client]

In [155]:
#Federated Training Loop
MAX_CLIENT_EPOCH = 2
MAX_FED_ROUNDS = 5
loss_fn = CrossEntropyLoss()

for r in range(MAX_FED_ROUNDS):
    fl_messages = []
    for c in clients:
        model = copy.copy(base_peft_model)
        set_peft_model_state_dict(model, global_adapters)

        optimizer = torch.optim.Adam(model.parameters(), lr=.01)

        for e in range(MAX_CLIENT_EPOCH):
            for batch in c.data_loader:

                x = batch["input_ids"]
                y = batch["labels"]

                model.train()
                resp_tokens = model(x)
                loss = loss_fn(resp_tokens, y)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        # send lora adapter
        round_response = FLMessage(
            client_name=c.client_name,
            num_samples=len(c.data_loader),
            lora_sd=get_peft_model_state_dict(model),
        )

        fl_messages.append(round_response)
        # Should send a message for each client per round
    global_adapters = fed_avg(fl_messages)

<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>


KeyboardInterrupt: 